# Response of the Thermosphere and Ionosphere to Geomagnetic Storms — Implementation / 구현

**Paper**: Fuller-Rowell, T. J., M. V. Codrescu, R. G. Roble, and A. D. Richmond (1994), *J. Geophys. Res.*, 99(A3), 3893–3914.

This notebook implements the key physical mechanisms described in the paper:
이 노트북은 논문에서 설명한 핵심 물리 메커니즘을 구현합니다:

1. **Scale height & composition vs. temperature** — 스케일 높이와 조성의 온도 의존성
2. **O/N₂ ratio change and NmF2 response** — O/N₂ 비율 변화와 NmF2 반응
3. **Joule heating rate estimation** — 줄 가열률 추정
4. **Wind-driven F-layer uplift** — 바람에 의한 F층 상승
5. **Composition bulge diurnal modulation** — 조성 돌출의 일주 변동 모사

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants
K_B = 1.381e-23       # Boltzmann constant [J/K]
M_U = 1.661e-27       # Atomic mass unit [kg]
G = 9.5               # Gravitational acceleration at ~300 km [m/s^2]
R_E = 6.371e6         # Earth radius [m]
OMEGA = 7.292e-5      # Earth rotation rate [rad/s]

# Atomic/molecular masses [amu]
M_O = 16.0            # Atomic oxygen
M_N2 = 28.0           # Molecular nitrogen
M_O2 = 32.0           # Molecular oxygen

## Part 1: Scale Height and Composition vs. Temperature / 스케일 높이와 조성의 온도 의존성

The paper's core mechanism: temperature increase → scale height increase → heavier species (N₂) extend to higher altitudes → O/N₂ ratio decreases at a fixed altitude.

논문의 핵심 메커니즘: 온도 상승 → 스케일 높이 증가 → 무거운 성분(N₂)이 더 높은 고도까지 확장 → 고정 고도에서 O/N₂ 비율 감소.

$$H_i = \frac{k_B T}{m_i g}$$

At a reference altitude $z_0$ with known densities, the density at altitude $z$ is:

$$n_i(z) = n_i(z_0) \exp\left(-\frac{z - z_0}{H_i}\right)$$

In [ ]:
def scale_height(T: float, m_amu: float) -> float:
    """Compute atmospheric scale height.

    Args:
        T: Temperature in Kelvin.
        m_amu: Molecular mass in atomic mass units.

    Returns:
        Scale height in km.
    """
    return K_B * T / (m_amu * M_U * G) / 1e3  # Convert m to km


def density_profile(z: np.ndarray, z0: float, n0: float,
                    T: float, m_amu: float) -> np.ndarray:
    """Compute number density vs. altitude assuming isothermal atmosphere.

    Args:
        z: Altitude array in km.
        z0: Reference altitude in km.
        n0: Number density at z0 in m^-3.
        T: Temperature in Kelvin.
        m_amu: Molecular mass in amu.

    Returns:
        Number density array in m^-3.
    """
    H = scale_height(T, m_amu)
    return n0 * np.exp(-(z - z0) / H)


# --- Plot scale heights vs. temperature ---
T_range = np.linspace(600, 2000, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scale heights
for m, label, color in [(M_O, 'O (16 amu)', 'C0'),
                         (M_N2, 'N₂ (28 amu)', 'C1'),
                         (M_O2, 'O₂ (32 amu)', 'C3')]:
    H = [scale_height(T, m) for T in T_range]
    axes[0].plot(T_range, H, label=label, color=color, linewidth=2)

# Mark quiet (1000 K) and storm (1400 K) conditions
for T_mark, label in [(1000, 'Quiet\n(1000 K)'), (1400, 'Storm\n(1400 K)')]:
    axes[0].axvline(T_mark, ls='--', color='gray', alpha=0.5)
    axes[0].text(T_mark + 20, 95, label, fontsize=9)

axes[0].set_xlabel('Exospheric Temperature [K]')
axes[0].set_ylabel('Scale Height [km]')
axes[0].set_title('Scale Height vs. Temperature / 스케일 높이 vs. 온도')
axes[0].legend()
axes[0].set_ylim(0, 110)

# Right: density profiles at quiet vs. storm temperatures
z = np.linspace(120, 500, 200)
z0 = 120  # Reference altitude [km]
n0_O = 5e17    # O density at 120 km [m^-3]
n0_N2 = 4e18   # N2 density at 120 km [m^-3]

T_quiet, T_storm = 1000, 1400

for T, ls, alpha in [(T_quiet, '-', 1.0), (T_storm, '--', 0.8)]:
    label_suffix = f' (T={T} K)'
    axes[1].semilogy(z, density_profile(z, z0, n0_O, T, M_O),
                     color='C0', ls=ls, alpha=alpha, linewidth=2,
                     label='O' + label_suffix)
    axes[1].semilogy(z, density_profile(z, z0, n0_N2, T, M_N2),
                     color='C1', ls=ls, alpha=alpha, linewidth=2,
                     label='N₂' + label_suffix)

axes[1].axhline(y=1e14, color='gray', ls=':', alpha=0.3)
axes[1].set_xlabel('Altitude [km]')
axes[1].set_ylabel('Number Density [m⁻³]')
axes[1].set_title('Density Profiles: Quiet vs. Storm / 밀도 프로파일: 정상 vs. 폭풍')
axes[1].legend(fontsize=8, ncol=2)
axes[1].set_xlim(120, 500)
axes[1].set_ylim(1e12, 1e19)

plt.tight_layout()
plt.show()

# Print key values
print("=== Scale Height Comparison / 스케일 높이 비교 ===")
for T in [1000, 1400]:
    print(f"\nT = {T} K:")
    for m, name in [(M_O, 'O'), (M_N2, 'N₂'), (M_O2, 'O₂')]:
        print(f"  H({name}) = {scale_height(T, m):.1f} km")

## Part 2: O/N₂ Ratio and NmF2 Response / O/N₂ 비율과 NmF2 반응

The paper demonstrates that the negative ionospheric storm effect is primarily caused by decreased O/N₂ ratio. Here we compute:

논문은 음의 전리층 폭풍 효과가 주로 O/N₂ 비율 감소에 의해 발생함을 보여줍니다. 여기서 계산하는 것:

1. O/N₂ column density ratio as a function of temperature change / 온도 변화의 함수로서 O/N₂ 칼럼 밀도 비율
2. Resulting NmF2 change using the approximation $N_mF2 \propto n(O)/n(N_2)$ / 근사식을 사용한 NmF2 변화

In [ ]:
def on2_ratio_at_altitude(z_km: float, T: float,
                          z0: float = 120.0,
                          n0_O: float = 5e17,
                          n0_N2: float = 4e18) -> float:
    """Compute O/N₂ number density ratio at a given altitude.

    Args:
        z_km: Target altitude in km.
        T: Exospheric temperature in K.
        z0: Reference altitude in km.
        n0_O: O density at z0 in m^-3.
        n0_N2: N2 density at z0 in m^-3.

    Returns:
        O/N₂ ratio (dimensionless).
    """
    n_O = density_profile(np.array([z_km]), z0, n0_O, T, M_O)[0]
    n_N2 = density_profile(np.array([z_km]), z0, n0_N2, T, M_N2)[0]
    return n_O / n_N2


# --- Compute O/N₂ ratio and NmF2 change vs. temperature increase ---
T_quiet = 1000  # K
delta_T = np.linspace(0, 600, 100)  # Temperature increase [K]
z_target = 300  # km (F2 peak altitude)

# O/N₂ ratio at 300 km for each temperature
on2_quiet = on2_ratio_at_altitude(z_target, T_quiet)
on2_storm = [on2_ratio_at_altitude(z_target, T_quiet + dT)
             for dT in delta_T]
on2_ratio_change = np.array(on2_storm) / on2_quiet

# NmF2 change (proportional to O/N₂)
nmf2_ratio = on2_ratio_change

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: O/N₂ ratio change
axes[0].plot(delta_T, on2_ratio_change, 'C0', linewidth=2)
axes[0].axhline(1.0, color='gray', ls='--', alpha=0.5)
axes[0].axvline(400, color='C3', ls=':', alpha=0.7,
                label='ΔT = 400 K (paper value)')
idx_400 = np.argmin(np.abs(delta_T - 400))
axes[0].plot(400, on2_ratio_change[idx_400], 'ro', markersize=10)
axes[0].annotate(f'{on2_ratio_change[idx_400]:.2f}',
                 (400, on2_ratio_change[idx_400]),
                 textcoords="offset points", xytext=(15, 10),
                 fontsize=12, color='C3')
axes[0].set_xlabel('Temperature Increase ΔT [K]')
axes[0].set_ylabel('O/N₂ Ratio (Storm / Quiet)')
axes[0].set_title(f'O/N₂ Ratio Change at {z_target} km / O/N₂ 비율 변화')
axes[0].legend()

# Right: NmF2 percentage change
nmf2_pct = (nmf2_ratio - 1) * 100
axes[1].plot(delta_T, nmf2_pct, 'C1', linewidth=2)
axes[1].axhline(0, color='gray', ls='--', alpha=0.5)
axes[1].axvline(400, color='C3', ls=':', alpha=0.7)
axes[1].fill_between(delta_T, nmf2_pct, 0, alpha=0.15, color='C1')
axes[1].plot(400, nmf2_pct[idx_400], 'ro', markersize=10)
axes[1].annotate(f'{nmf2_pct[idx_400]:.0f}%',
                 (400, nmf2_pct[idx_400]),
                 textcoords="offset points", xytext=(15, 10),
                 fontsize=12, color='C3')
axes[1].set_xlabel('Temperature Increase ΔT [K]')
axes[1].set_ylabel('NmF2 Change [%]')
axes[1].set_title('Estimated NmF2 Change / NmF2 변화 추정')

plt.tight_layout()
plt.show()

print(f"At ΔT = 400 K (paper value):")
print(f"  O/N₂ ratio drops to {on2_ratio_change[idx_400]:.2f}x of quiet value")
print(f"  NmF2 decreases by ~{-nmf2_pct[idx_400]:.0f}%")
print(f"  (Paper reports up to ~45% decrease in NmF2, consistent with"
      f" composition + wind effects combined)")

## Part 3: Joule Heating Rate / 줄 가열률

The paper uses enhanced cross-polar-cap potential (45 kV → 130 kV) during storms. Joule heating is the primary energy source driving thermospheric response.

논문은 폭풍 시 cross-polar-cap potential이 45 kV에서 130 kV로 증가합니다. 줄 가열은 열권 반응을 구동하는 주요 에너지원입니다.

$$Q_J = \sigma_P |\mathbf{E} + \mathbf{V}_n \times \mathbf{B}|^2$$

We estimate Joule heating for quiet vs. storm conditions at different latitudes.
정상 상태와 폭풍 상태에서 위도별 줄 가열을 추정합니다.

In [ ]:
def joule_heating_rate(E_mVm: float, sigma_P: float,
                       Vn: float = 0.0, B: float = 5e-5) -> float:
    """Compute height-integrated Joule heating rate.

    Args:
        E_mVm: Electric field magnitude in mV/m.
        sigma_P: Height-integrated Pedersen conductivity in S.
        Vn: Neutral wind speed perpendicular to B in m/s.
        B: Magnetic field strength in T.

    Returns:
        Joule heating rate in mW/m^2.
    """
    E_eff = E_mVm * 1e-3 + Vn * B  # Effective electric field [V/m]
    return sigma_P * E_eff**2 * 1e3  # Convert W/m^2 to mW/m^2


def electric_field_from_potential(phi_kV: float,
                                 polar_cap_radius_deg: float = 15.0) -> float:
    """Estimate average electric field from cross-polar-cap potential.

    Args:
        phi_kV: Cross-polar-cap potential drop in kV.
        polar_cap_radius_deg: Polar cap radius in degrees.

    Returns:
        Average electric field in mV/m.
    """
    R_pc = R_E * np.radians(polar_cap_radius_deg)  # Polar cap radius [m]
    E = phi_kV * 1e3 / (2 * R_pc)  # V/m
    return E * 1e3  # mV/m


# --- Compare quiet vs. storm Joule heating ---
phi_quiet = 45    # kV (paper value)
phi_storm = 130   # kV (paper value)

E_quiet = electric_field_from_potential(phi_quiet)
E_storm = electric_field_from_potential(phi_storm)

# Pedersen conductivity range (height-integrated)
sigma_P_range = np.linspace(1, 20, 100)  # S (Siemens)

Q_quiet = [joule_heating_rate(E_quiet, s) for s in sigma_P_range]
Q_storm = [joule_heating_rate(E_storm, s) for s in sigma_P_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Joule heating vs. Pedersen conductivity
axes[0].plot(sigma_P_range, Q_quiet, 'C0', linewidth=2,
             label=f'Quiet (Φ={phi_quiet} kV)')
axes[0].plot(sigma_P_range, Q_storm, 'C3', linewidth=2,
             label=f'Storm (Φ={phi_storm} kV)')
axes[0].axhline(50, color='gray', ls=':', alpha=0.5,
                label='Paper max: 50 mW/m²')
axes[0].set_xlabel('Height-Integrated Pedersen Conductivity [S]')
axes[0].set_ylabel('Joule Heating Rate [mW/m²]')
axes[0].set_title('Joule Heating: Quiet vs. Storm / 줄 가열: 정상 vs. 폭풍')
axes[0].legend()
axes[0].set_ylim(0, 80)

# Right: Heating rate vs. latitude (simplified model)
lat = np.linspace(50, 90, 100)
# Simple model: E peaks in auroral oval (~65-75 deg)
E_lat_quiet = E_quiet * np.exp(-0.5 * ((lat - 70) / 5)**2)
E_lat_storm = E_storm * np.exp(-0.5 * ((lat - 67) / 7)**2)

sigma_P_typical = 8  # S (typical for active auroral conditions)
Q_lat_quiet = [joule_heating_rate(E, sigma_P_typical) for E in E_lat_quiet]
Q_lat_storm = [joule_heating_rate(E, sigma_P_typical) for E in E_lat_storm]

axes[1].plot(lat, Q_lat_quiet, 'C0', linewidth=2, label='Quiet')
axes[1].plot(lat, Q_lat_storm, 'C3', linewidth=2, label='Storm')
axes[1].fill_between(lat, Q_lat_storm, Q_lat_quiet, alpha=0.15, color='C3',
                     label='Storm enhancement')
axes[1].set_xlabel('Geographic Latitude [°N]')
axes[1].set_ylabel('Joule Heating Rate [mW/m²]')
axes[1].set_title('Latitude Profile of Joule Heating / 위도별 줄 가열')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"=== Electric Field Estimates / 전기장 추정 ===")
print(f"Quiet: E = {E_quiet:.1f} mV/m (Φ = {phi_quiet} kV)")
print(f"Storm: E = {E_storm:.1f} mV/m (Φ = {phi_storm} kV)")
print(f"Storm/Quiet ratio: {(phi_storm/phi_quiet)**2:.1f}x in heating rate"
      f" (since Q ∝ E²)")

## Part 4: Wind-Driven F-Layer Uplift / 바람에 의한 F층 상승

Equatorward meridional winds push the F-layer to higher altitudes via the relation:

적도 방향 자오선 풍은 다음 관계를 통해 F층을 높은 고도로 밀어 올립니다:

$$\frac{dh_F}{dt} = V_n \sin I \cos I$$

At higher altitudes, the loss rate decreases (less N₂), producing a **positive storm effect**.
더 높은 고도에서는 손실률이 감소(N₂ 감소)하여 **양의 폭풍 효과**를 생성합니다.

In [ ]:
def magnetic_dip_angle(lat_deg: float) -> float:
    """Compute magnetic dip angle from geographic latitude (dipole approx).

    Args:
        lat_deg: Geographic latitude in degrees.

    Returns:
        Dip angle in radians.
    """
    lat_rad = np.radians(lat_deg)
    return np.arctan(2 * np.tan(lat_rad))


def f_layer_uplift_rate(Vn: float, lat_deg: float) -> float:
    """Compute F-layer height change rate due to meridional wind.

    Args:
        Vn: Equatorward meridional wind speed in m/s (positive = equatorward).
        lat_deg: Geographic latitude in degrees.

    Returns:
        dh/dt in m/s.
    """
    I = magnetic_dip_angle(lat_deg)
    return Vn * np.sin(I) * np.cos(I)


def nmf2_change_from_uplift(dh_km: float, T: float = 1000.0) -> float:
    """Estimate NmF2 change factor from F-layer height change.

    The loss rate L ∝ n(N₂) ∝ exp(-dh/H_N2), so NmF2 ∝ 1/L increases.

    Args:
        dh_km: Height change in km (positive = upward).
        T: Exospheric temperature in K.

    Returns:
        NmF2 ratio (storm/quiet).
    """
    H_N2 = scale_height(T, M_N2)
    return np.exp(dh_km / H_N2)


# --- Wind uplift effect across latitudes ---
latitudes = np.linspace(20, 70, 100)
Vn_values = [50, 100, 200]  # Equatorward wind speeds [m/s]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: Uplift rate vs. latitude
for Vn in Vn_values:
    uplift = [f_layer_uplift_rate(Vn, lat) for lat in latitudes]
    axes[0].plot(latitudes, uplift, linewidth=2,
                 label=f'Vn = {Vn} m/s')
axes[0].set_xlabel('Geographic Latitude [°]')
axes[0].set_ylabel('dh/dt [m/s]')
axes[0].set_title('F-Layer Uplift Rate / F층 상승률')
axes[0].legend()

# Middle: Height change after 3 hours
duration_s = 3 * 3600  # 3 hours in seconds
for Vn in Vn_values:
    dh = [f_layer_uplift_rate(Vn, lat) * duration_s / 1e3
          for lat in latitudes]  # km
    axes[1].plot(latitudes, dh, linewidth=2, label=f'Vn = {Vn} m/s')
axes[1].set_xlabel('Geographic Latitude [°]')
axes[1].set_ylabel('Δh [km] after 3 hours')
axes[1].set_title('F-Layer Height Change (3 hr) / F층 높이 변화 (3시간)')
axes[1].legend()

# Right: NmF2 increase from uplift (positive storm effect)
for Vn in Vn_values:
    dh = [f_layer_uplift_rate(Vn, lat) * duration_s / 1e3
          for lat in latitudes]
    nmf2_change = [(nmf2_change_from_uplift(h) - 1) * 100 for h in dh]
    axes[2].plot(latitudes, nmf2_change, linewidth=2,
                 label=f'Vn = {Vn} m/s')
axes[2].set_xlabel('Geographic Latitude [°]')
axes[2].set_ylabel('NmF2 Change [%]')
axes[2].set_title('Positive Storm Effect from Uplift / 상승에 의한 양의 폭풍 효과')
axes[2].axhline(0, color='gray', ls='--', alpha=0.5)
axes[2].legend()

plt.tight_layout()
plt.show()

# Worked example at 45° latitude
lat_ex = 45
Vn_ex = 100  # m/s (paper value for storm wind at mid-latitudes)
I_ex = np.degrees(magnetic_dip_angle(lat_ex))
uplift_ex = f_layer_uplift_rate(Vn_ex, lat_ex)
dh_3hr = uplift_ex * 3 * 3600 / 1e3  # km
nmf2_ex = nmf2_change_from_uplift(dh_3hr)

print(f"\n=== Worked Example at {lat_ex}°N / {lat_ex}°N에서의 계산 예 ===")
print(f"Dip angle I = {I_ex:.1f}°")
print(f"Equatorward wind Vn = {Vn_ex} m/s")
print(f"Uplift rate = {uplift_ex:.1f} m/s = {uplift_ex*3600/1e3:.1f} km/hr")
print(f"Height change after 3 hr = {dh_3hr:.1f} km")
print(f"NmF2 increase = {(nmf2_ex-1)*100:.0f}% (positive storm effect)")

## Part 5: Composition Bulge Diurnal Modulation / 조성 돌출의 일주 변동 모사

The paper's key insight: the composition bulge does **not corotate** with Earth. On the nightside, background winds (equatorward) assist bulge expansion; on the dayside, background winds (poleward) restrict it.

논문의 핵심 통찰: 조성 돌출은 지구와 함께 **회전하지 않습니다**. 야간에는 배경풍(적도 방향)이 돌출 확장을 도우며, 주간에는 배경풍(극 방향)이 이를 억제합니다.

We simulate a simplified 1D model of composition bulge latitude vs. local time.
조성 돌출 위도의 local time 의존성을 간단한 1D 모델로 시뮬레이션합니다.

In [ ]:
def background_meridional_wind(lt_hr: float, lat_deg: float) -> float:
    """Model quiet-time meridional wind (positive = equatorward).

    Daytime: poleward (negative), Nighttime: equatorward (positive).
    Based on typical thermospheric wind patterns at equinox.

    Args:
        lt_hr: Local time in hours (0-24).
        lat_deg: Latitude in degrees.

    Returns:
        Meridional wind in m/s (positive = equatorward).
    """
    # Simple sinusoidal model: peak equatorward at ~02 LT, peak poleward at ~14 LT
    phase = 2 * np.pi * (lt_hr - 2) / 24
    amplitude = 50 + 0.5 * lat_deg  # Stronger at higher latitudes
    return amplitude * np.cos(phase)


def storm_wind(t_storm_hr: float, lat_deg: float,
               storm_duration: float = 12.0) -> float:
    """Model storm-driven equatorward wind.

    Args:
        t_storm_hr: Time since storm onset in hours.
        lat_deg: Latitude in degrees.
        storm_duration: Storm duration in hours.

    Returns:
        Storm-driven equatorward wind in m/s.
    """
    if t_storm_hr < 0 or t_storm_hr > storm_duration:
        return 0.0
    # Ramp up over 3 hours, sustain, then drop
    if t_storm_hr < 3:
        ramp = t_storm_hr / 3
    elif t_storm_hr > storm_duration - 1:
        ramp = (storm_duration - t_storm_hr)
    else:
        ramp = 1.0
    # Amplitude decreases with lower latitude (surge weakens)
    lat_factor = np.clip((lat_deg - 20) / 50, 0, 1)
    return 150 * ramp * lat_factor  # Peak ~150 m/s at high latitudes


def simulate_bulge_boundary(storm_start_ut: float = 12.0,
                            storm_duration: float = 12.0,
                            total_hours: float = 36.0,
                            dt_hr: float = 0.25) -> dict:
    """Simulate the equatorward boundary of the composition bulge.

    For each longitude sector, track how far equatorward the bulge penetrates,
    considering both storm and background winds.

    Args:
        storm_start_ut: UT of storm onset in hours.
        storm_duration: Duration of storm in hours.
        total_hours: Total simulation time in hours.
        dt_hr: Time step in hours.

    Returns:
        Dictionary with time array and bulge boundary for each longitude.
    """
    n_steps = int(total_hours / dt_hr)
    times = np.arange(n_steps) * dt_hr

    # Track 4 longitude sectors (0, 90, 180, 270 deg)
    longitudes = [0, 90, 180, 270]
    results = {}

    for lon in longitudes:
        bulge_lat = np.full(n_steps, 70.0)  # Start at 70° (auroral zone)

        for i in range(1, n_steps):
            ut = times[i]
            lt = (ut + lon / 15) % 24  # Local time
            t_storm = ut - storm_start_ut  # Time since storm onset

            current_lat = bulge_lat[i - 1]

            # Total wind = background + storm
            v_bg = background_meridional_wind(lt, current_lat)
            v_storm = storm_wind(t_storm, current_lat, storm_duration)
            v_total = v_bg + v_storm

            # Convert wind to latitude change
            # Positive wind = equatorward = decreasing latitude
            dlat = -v_total * dt_hr * 3600 / (R_E * np.pi / 180)

            # Diffusion restores toward 70° (relaxation)
            tau_relax = 24.0  # hours
            dlat_relax = (70.0 - current_lat) * dt_hr / tau_relax

            bulge_lat[i] = np.clip(current_lat + dlat + dlat_relax, 20, 75)

        results[lon] = bulge_lat

    return {'time': times, 'longitudes': longitudes, 'bulge': results}


# --- Run simulation for GS1 (storm at 1200 UT) ---
sim = simulate_bulge_boundary(storm_start_ut=12.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Bulge boundary vs. time for each longitude
colors = ['C0', 'C1', 'C2', 'C3']
for lon, color in zip(sim['longitudes'], colors):
    lt_at_storm = (12 + lon / 15) % 24
    label = f'{lon}°E (LT={lt_at_storm:.0f}h at storm onset)'
    axes[0].plot(sim['time'], sim['bulge'][lon], color=color,
                 linewidth=2, label=label)

axes[0].axvspan(12, 24, alpha=0.1, color='gray', label='Storm period')
axes[0].set_xlabel('UT [hours]')
axes[0].set_ylabel('Bulge Equatorward Boundary [°N]')
axes[0].set_title('Composition Bulge Migration / 조성 돌출 이동\n(GS1: Storm at 1200 UT)')
axes[0].legend(fontsize=8)
axes[0].set_ylim(30, 75)
axes[0].invert_yaxis()

# Right: Polar plot showing bulge at different times
ax_polar = fig.add_subplot(122, projection='polar')
times_to_show = [6, 12, 18, 24]  # Hours after storm onset
colors_time = ['C0', 'C4', 'C3', 'C1']

for t_show, color in zip(times_to_show, colors_time):
    # Interpolate bulge boundary at many longitudes
    lons_fine = np.linspace(0, 360, 73)
    bulge_lats = []
    for lon_f in lons_fine:
        # Interpolate from 4 sectors
        lon_f_mod = lon_f % 360
        lon_weights = []
        for lon_s in sim['longitudes']:
            dist = min(abs(lon_f_mod - lon_s), 360 - abs(lon_f_mod - lon_s))
            lon_weights.append(np.exp(-dist**2 / (90**2)))
        lon_weights = np.array(lon_weights)
        lon_weights /= lon_weights.sum()

        idx = int(t_show / (sim['time'][1] - sim['time'][0])) + int(12 / (sim['time'][1] - sim['time'][0]))
        idx = min(idx, len(sim['time']) - 1)

        lat_val = sum(w * sim['bulge'][lon][idx]
                      for w, lon in zip(lon_weights, sim['longitudes']))
        bulge_lats.append(lat_val)

    theta = np.radians(lons_fine)
    r = 90 - np.array(bulge_lats)  # Convert: 90° lat = center
    ax_polar.plot(theta, r, color=color, linewidth=2,
                  label=f't = {t_show}h after onset')

ax_polar.set_theta_zero_location('N')  # 0° at top
ax_polar.set_theta_direction(-1)  # Clockwise
ax_polar.set_rlabel_position(45)
ax_polar.set_rticks([10, 20, 30, 40])
ax_polar.set_yticklabels(['80°', '70°', '60°', '50°'])
ax_polar.set_title('Bulge Boundary (North Pole view)\n조성 돌출 경계 (북극 조감도)',
                    pad=20)
ax_polar.legend(loc='lower left', bbox_to_anchor=(-0.1, -0.15), fontsize=8)

plt.tight_layout()
plt.show()

print("=== Key Finding / 핵심 발견 ===")
print("The composition bulge penetrates deeper equatorward on the nightside")
print("(where background wind assists) and is restricted on the dayside")
print("(where background wind opposes storm wind).")
print("조성 돌출은 야간(배경풍이 도움)에서 더 깊이 적도로 침투하고,")
print("주간(배경풍이 반대)에서는 억제됩니다.")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Thermosphere-ionosphere model / 열권-전리층 모델 | CTIM (15 pressure levels, 2°×18° resolution) | TIE-GCM, WACCM-X (higher resolution, more physics) |
| Storm forcing / 폭풍 강제 | Cross-polar-cap potential 45→130 kV, idealized 12-hr step | AMIE-driven realistic time-varying inputs |
| Composition diagnostic / 조성 진단 | Mean molecular mass (m) at pressure level 12 | GUVI/TIMED O/N₂ column density ratio observations |
| Ionospheric response / 전리층 반응 | NmF2 change from coupled model | GPS TEC maps, ionosonde networks, COSMIC RO |
| Positive storm effect / 양의 폭풍 효과 | Wind-driven F-layer uplift | Same mechanism, now measured by digisonde hmF2 |
| Negative storm effect / 음의 폭풍 효과 | O/N₂ decrease from composition bulge | GUVI confirms; used operationally in SWPC forecasts |
| UT dependence / UT 의존성 | 4 simulations at different UT start times | Now understood as geographic-geomagnetic offset effect |